# Housing Price Prediction: From Traditional ML to Fine-Tuned LLMs

This notebook demonstrates Week 6 concepts from the LLM Engineering course:
- Data curation and preprocessing
- Evaluation framework with W&B integration
- Traditional ML baselines (Linear Regression, Random Forest, XGBoost)
- Neural network with PyTorch
- Zero-shot LLM comparison
- Fine-tuning LLaMA 3.2 3B with QLoRA

**Dataset:** USA Real Estate (Kaggle)
**W&B Project:** `llama-3.2-housing-pricer`

## Step 0: Setup and Imports

In [ ]:
# Uncomment to install dependencies
!uv pip install kagglehub pandas numpy scikit-learn xgboost torch litellm wandb tqdm

In [ ]:
import warnings
warnings.filterwarnings('ignore')

from tqdm import tqdm

from data import load_raw_data, filter_data, create_splits, PRICE_MAX, PRICE_MIN
from evaluation import init_wandb, log_model_results, log_training_step, create_comparison_table, mean_squared_error, mean_absolute_error, create_comparison_charts, compute_metrics
from models import RandomPricer, MeanPricer, train_random_forest, train_xgboost, train_neural_network, train_linear_regression, predict_neural_network, predict_with_llm, parse_price

from prompts import create_description, format_for_inference, format_for_finetuning

## Step 1: Load and Explore Data

In [ ]:
df_raw = load_raw_data()
print(f"Raw data: {len(df_raw):,} rows")
df_raw.head()

In [ ]:
print("\nColumn info:")
print(df_raw.dtypes)
print(f"\nPrice range: ${df_raw['price'].min():,.0f} - ${df_raw['price'].max():,.0f}")

## Step 2: Filter and Split Data

In [ ]:
df_clean = filter_data(df_raw)
print(f"After filtering (${PRICE_MIN:,} - ${PRICE_MAX:,}): {len(df_clean):,} rows")

In [ ]:
train_df, val_df, test_df = create_splits(df_clean, train_size=500, val_size=50, test_size=50)
print(f"Train: {len(train_df)}, Val: {len(val_df)}, Test: {len(test_df)}")

## Step 3: Create Text Descriptions

Convert structured features to natural language for LLM-based pricing.

In [ ]:
train_df = train_df.copy()
val_df = val_df.copy()
test_df = test_df.copy()

train_df['description'] = train_df.apply(create_description, axis=1)
val_df['description'] = val_df.apply(create_description, axis=1)
test_df['description'] = test_df.apply(create_description, axis=1)

In [ ]:
print("Example description:")
print(train_df['description'].iloc[0])
print(f"\nActual price: ${train_df['price'].iloc[0]:,.0f}")

## Step 4: Initialize W&B

All experiments will be logged to the `llama-3.2-housing-pricer` project.

In [ ]:
import wandb
wandb.login()

In [ ]:
init_wandb(
    config={
        "dataset": "usa-real-estate",
        "train_size": len(train_df),
        "val_size": len(val_df),
        "test_size": len(test_df),
        "price_range": f"${PRICE_MIN:,}-${PRICE_MAX:,}"
    },
    run_name="house-pricer-comparison"
)

## Step 5: Baseline Models

Start with simple baselines to establish a floor for comparison.

In [ ]:
results = []
y_test = test_df['price'].values

In [ ]:
# Random baseline
random_pricer = RandomPricer(PRICE_MIN, PRICE_MAX)
random_preds = random_pricer.predict(test_df)
metrics = log_model_results("Random", y_test, random_preds)
results.append({"name": "Random", "type": "Baseline", **metrics})
print(f"Random Baseline - MAE: ${metrics['mae']:,.0f}, RMSE: ${metrics['rmse']:,.0f}, MAPE: {metrics['mape']:.1f}%")

In [ ]:
# Mean baseline
mean_pricer = MeanPricer()
mean_pricer.fit(train_df['price'].values)
mean_preds = mean_pricer.predict(test_df)
metrics = log_model_results("Mean", y_test, mean_preds)
results.append({"name": "Mean", "type": "Baseline", **metrics})
print(f"Mean Baseline - MAE: ${metrics['mae']:,.0f}, RMSE: ${metrics['rmse']:,.0f}, MAPE: {metrics['mape']:.1f}%")

## Step 6: Traditional ML Models

Use structured features (bed, bath, sqft, acres) for traditional ML.

In [ ]:
features = ['bed', 'bath', 'house_size', 'acre_lot']
X_train = train_df[features].fillna(0).values
X_val = val_df[features].fillna(0).values
X_test = test_df[features].fillna(0).values
y_train = train_df['price'].values
y_val = val_df['price'].values

In [ ]:
# Linear Regression
lr_model = train_linear_regression(X_train, y_train)
lr_preds = lr_model.predict(X_test)
metrics = log_model_results("LinearRegression", y_test, lr_preds)
results.append({"name": "LinearRegression", "type": "Traditional ML", **metrics})
print(f"Linear Regression - MAE: ${metrics['mae']:,.0f}, RMSE: ${metrics['rmse']:,.0f}, MAPE: {metrics['mape']:.1f}%")

In [ ]:
# Random Forest
rf_model = train_random_forest(X_train, y_train)
rf_preds = rf_model.predict(X_test)
metrics = log_model_results("RandomForest", y_test, rf_preds)
results.append({"name": "RandomForest", "type": "Traditional ML", **metrics})
print(f"Random Forest - MAE: ${metrics['mae']:,.0f}, RMSE: ${metrics['rmse']:,.0f}, MAPE: {metrics['mape']:.1f}%")

In [ ]:
# XGBoost
xgb_model = train_xgboost(X_train, y_train)
xgb_preds = xgb_model.predict(X_test)
metrics = log_model_results("XGBoost", y_test, xgb_preds)
results.append({"name": "XGBoost", "type": "Traditional ML", **metrics})
print(f"XGBoost - MAE: ${metrics['mae']:,.0f}, RMSE: ${metrics['rmse']:,.0f}, MAPE: {metrics['mape']:.1f}%")

## Step 7: Neural Network

Train a simple MLP on the structured features.

In [ ]:
nn_model = train_neural_network(
    X_train, y_train,
    input_dim=4,
    epochs=100,
    learning_rate=1e-3,
    log_fn=log_training_step
)

In [ ]:
nn_preds = predict_neural_network(nn_model, X_test)
metrics = log_model_results("NeuralNet", y_test, nn_preds)
results.append({"name": "NeuralNet", "type": "Deep Learning", **metrics})
print(f"Neural Network - MAE: ${metrics['mae']:,.0f}, RMSE: ${metrics['rmse']:,.0f}, MAPE: {metrics['mape']:.1f}%")

## Step 8: Zero-Shot LLM Comparison

Test frontier LLMs on price estimation without fine-tuning.

**Note:** This uses a sample of 10 test examples to manage API costs.

In [ ]:
sample_test = test_df.head(10)
sample_y_test = sample_test['price'].values

In [ ]:
# GPT-4o-mini
gpt_preds = []
for _, row in tqdm(sample_test.iterrows(), total=len(sample_test), desc="GPT-4o-mini"):
    messages = format_for_inference(row['description'])
    pred = predict_with_llm("gpt-4o-mini", messages)
    gpt_preds.append(pred)

metrics = log_model_results("GPT-4o-mini", sample_y_test, gpt_preds)
results.append({"name": "GPT-4o-mini (zero-shot)", "type": "LLM", **metrics})
print(f"GPT-4o-mini - MAE: ${metrics['mae']:,.0f}, RMSE: ${metrics['rmse']:,.0f}, MAPE: {metrics['mape']:.1f}%")

In [ ]:
# Claude Haiku (optional - uncomment to run)
# claude_preds = []
# for _, row in tqdm(sample_test.iterrows(), total=len(sample_test), desc="Claude Haiku"):
#     messages = format_for_inference(row['description'])
#     pred = predict_with_llm("claude-3-haiku-20240307", messages)
#     claude_preds.append(pred)

# metrics = log_model_results("Claude-Haiku", sample_y_test, claude_preds)
# results.append({"name": "Claude-Haiku (zero-shot)", "type": "LLM", **metrics})
# print(f"Claude Haiku - MAE: ${metrics['mae']:,.0f}, RMSE: ${metrics['rmse']:,.0f}, MAPE: {metrics['mape']:.1f}%")

## Step 9: Fine-Tuning an OpenAI Model

In [ ]:
# Save training data for fine-tuning
finetune_data = train_df.apply(format_for_finetuning, axis=1).tolist()
validation_data = val_df.apply(format_for_finetuning, axis=1).tolist()
test_data = test_df.apply(format_for_finetuning, axis=1).tolist()

import json
with open('housing_pricer_train_finetune.jsonl', 'w') as f:
    for item in finetune_data:
        f.write(json.dumps(item) + '\n')

print(f"Saved {len(finetune_data)} training examples to housing_pricer_train_finetune.jsonl")

with open('housing_pricer_validation.jsonl', 'w') as f:
    for item in validation_data:
        f.write(json.dumps(item) + '\n')
print(f"Saved {len(validation_data)} validation examples to housing_pricer_validation.jsonl")

with open('housing_pricer_test.jsonl', 'w') as f:
    for item in test_data:
        f.write(json.dumps(item) + '\n')
print(f"Saved {len(test_data)} test examples to housing_pricer_test.jsonl")

In [ ]:
# Preview a training example
print("Training example format:")
print(finetune_data[0])

### Set up

In [ ]:
from openai import OpenAI
import dotenv

dotenv.load_dotenv(override=True)

open_ai = OpenAI()

### Upload training files to OpenAI

In [ ]:
with open("housing_pricer_train_finetune.jsonl", "rb") as f:
    train_file = open_ai.files.create(file=f, purpose="fine-tune")
train_file

In [ ]:
with open("housing_pricer_validation.jsonl", "rb") as f:
    validation_file = open_ai.files.create(file=f, purpose="fine-tune")
validation_file

### Fine Tune

In [ ]:
open_ai.fine_tuning.jobs.create(
    training_file=train_file.id,
    validation_file=validation_file.id,
    model="gpt-4.1-nano-2025-04-14",
    seed=42,
    hyperparameters={"n_epochs": 1, "batch_size": 1},
    suffix="housing_pricer"
)

In [ ]:
open_ai.fine_tuning.jobs.list(limit=1)

In [ ]:
job_id = open_ai.fine_tuning.jobs.list(limit=1).data[0].id

In [ ]:
job_id

In [ ]:
open_ai.fine_tuning.jobs.retrieve(job_id)

In [ ]:
open_ai.fine_tuning.jobs.list_events(fine_tuning_job_id=job_id, limit=10).data

### Test our fine tuned model


In [ ]:
fine_tuned_model_name = open_ai.fine_tuning.jobs.retrieve(job_id).fine_tuned_model

In [ ]:
print(fine_tuned_model_name)
# ft:gpt-4.1-nano-2025-04-14:personal:pricer:DHZCVlGd

In [ ]:
fine_tuned_preds = []
for _, row in tqdm(sample_test.iterrows(), total=len(sample_test), desc="Fine-tuned GPT"):
    messages = format_for_inference(row['description'])
    response = open_ai.chat.completions.create(
        model=fine_tuned_model_name,
        messages=messages,
        max_tokens=7
    )
    pred = parse_price(response.choices[0].message.content)
    fine_tuned_preds.append(pred)

metrics = log_model_results("Fine-tuned GPT", sample_y_test, gpt_preds)
results.append({"name": "Fine-tuned GPT", "type": "LLM", **metrics})
print(f"Fine-tuned GPT - MAE: ${metrics['mae']:,.0f}, RMSE: ${metrics['rmse']:,.0f}, MAPE: {metrics['mape']:.1f}%")

## Step 10: Final Comparison

In [ ]:
create_comparison_table(results)

In [ ]:
print("\n" + "="*60)
print("MODEL COMPARISON SUMMARY")
print("="*60)
print(f"{'Model':<30} {'Type':<15} {'MAE':>12} {'MAPE':>8}")
print("-"*60)
for r in sorted(results, key=lambda x: x['mae']):
    print(f"{r['name']:<30} {r['type']:<15} ${r['mae']:>10,.0f} {r['mape']:>7.1f}%")
print("="*60)

In [ ]:
create_comparison_charts(results)

In [ ]:
wandb.finish()
print("\nW&B run finished. View your results at: https://wandb.ai/")

## Pickle the results array so that I can reuse it in week 7

In [ ]:
import pickle

with open('comparison_results.pkl', 'wb') as f:
    pickle.dump(results, f)